# Topic: Appending Files - Append flags (a, a+), write cursor forcing, and atomic operations
 
## 1. THE APPEND MODE FLAGS
- **`a` (Append-Only)**: Opens a file to write data at the end of the file. Creates a new file if it does not exist.
- **`a+` (Append-Read)**: Opens a file for both reading and appending.
 
## 2. WRITE CURSOR FORCE BEHAVIOR
- In append mode, CPython enforces a strict rule: **all writes occur at the end of the file**.
  - Even if you call `f.seek(0)` to move the read cursor back to the beginning of the file, any subsequent write operations will immediately shift the file pointer to the end of the file before executing, preventing overwriting existing data.
 
## 3. ATOMIC POSIX CONCURRENCY
- On POSIX-compliant systems (like macOS, Linux), opening a file in append mode (`a`) translates to opening the file descriptor with the `O_APPEND` flag at the OS kernel level.
- This makes writes **atomic**: if multiple processes append to the same file concurrently, the OS guarantees that writes do not overlap or corrupt each other's data block.
 
---


In [ ]:
import os

append_demo_file = "append_demo.txt"

# 1. Create file with initial content
with open(append_demo_file, "w", encoding="utf-8") as f:
    f.write("Initial Line.\n")

# 2. Append new content
print("--- Standard Append ('a' Mode) ---")
with open(append_demo_file, "a", encoding="utf-8") as f:
    f.write("Appended Line 1.\n")
    f.write("Appended Line 2.\n")

with open(append_demo_file, "r", encoding="utf-8") as f:
    print("Content after append:")
    print(f.read())



In [ ]:
# 3. Seeking vs. Writing in Append Mode ('a+' Mode)
print("\n--- Testing Seek Write in Append Mode ---")
with open(append_demo_file, "a+", encoding="utf-8") as f:
    # Move pointer to beginning to read content
    f.seek(0)
    print(f"Reading after seek(0): {repr(f.readline().strip())}")
    
    # Try to write after seeking to start
    f.seek(0)  # Move pointer back to start
    f.write("Inserted Line?\n")  # This will be forced to the tail!
    
    # Read entire file again
    f.seek(0)
    print("\nFile contents after seek-write attempt:")
    print(f.read())
    # Expected: "Inserted Line?" is at the end, not the beginning!



In [ ]:
# 4. Clean up
if os.path.exists(append_demo_file):
    os.remove(append_demo_file)
